
# County Rainfall and Planting Advisory

## Project Overview

This project analyses monthly rainfall data from five Kenyan counties: Nairobi, Nakuru, Kisumu, Mombasa and Machakos.

The aim is to clean the rainfall data, use NumPy to calculate rainfall patterns, identify dry months and recommend two-month planting windows.

## Tools Used

- Python
- NumPy
- CSV module
- Google Colab


In [1]:

import numpy as np
import csv

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

print(counties)


['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']


In [3]:

clean_rows = [

    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi

    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru

    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu

    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]   # Mombasa

]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("First four counties:", len(clean_rows))
print("Machakos raw readings:", len(machakos_raw))


First four counties: 4
Machakos raw readings: 12


In [4]:

def clean_machakos(raw_data):

    valid_values = []
    invalid_months = []

    for index, value in enumerate(raw_data):

        try:
            number = int(value.strip())
            valid_values.append(number)

        except ValueError:
            invalid_months.append(index)

    average = round(sum(valid_values) / len(valid_values))

    cleaned_data = []

    for index, value in enumerate(raw_data):

        try:
            number = int(value.strip())
            cleaned_data.append(number)

        except ValueError:
            cleaned_data.append(average)

    return cleaned_data, invalid_months, average


machakos_cleaned, imputed_months, machakos_average = clean_machakos(machakos_raw)

print("Cleaned Machakos data:", machakos_cleaned)
print("Average used for replacement:", machakos_average)
print("Months imputed:", [month + 1 for month in imputed_months])


Cleaned Machakos data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Average used for replacement: 68
Months imputed: [4, 9]


In [5]:

# Combine the four clean counties with Machakos
rainfall = np.array(clean_rows + [machakos_cleaned])

# Display the rainfall data
print("Rainfall data:")
print(rainfall)

# Check the shape and data type
print("Array shape:", rainfall.shape)
print("Array data type:", rainfall.dtype)


Rainfall data:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array data type: int64


In [6]:

# Calculate annual rainfall for each county
annual_totals = np.sum(rainfall, axis=1)

# Display each county and its annual rainfall
for county, total in zip(counties, annual_totals):
    print(county, ":", total, "mm")


Nairobi : 1045 mm
Nakuru : 916 mm
Kisumu : 1371 mm
Mombasa : 1075 mm
Machakos : 811 mm


In [7]:

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

# Calculate average rainfall for each month
monthly_averages = np.mean(rainfall, axis=0)

# Display the monthly averages
for month, average in zip(months, monthly_averages):
    print(month, ":", round(average, 2), "mm")


January : 47.0 mm
February : 47.6 mm
March : 91.0 mm
April : 164.4 mm
May : 171.2 mm
June : 67.6 mm
July : 56.6 mm
August : 58.2 mm
September : 64.2 mm
October : 67.8 mm
November : 127.4 mm
December : 80.6 mm


In [8]:

# Find the position of the wettest month for each county
wettest_indices = np.argmax(rainfall, axis=1)

# Get the wettest month names
wettest_months = np.array(months)[wettest_indices]

# Display the results
for county, month in zip(counties, wettest_months):
    print(county, ":", month)


Nairobi : April
Nakuru : April
Kisumu : April
Mombasa : May
Machakos : November


In [9]:

# Create a Boolean mask for rainfall below 50 mm
dry_mask = rainfall < 50

# Count the dry months for each county
dry_month_counts = np.sum(dry_mask, axis=1)

# Display the results
for county, count in zip(counties, dry_month_counts):
    print(county, ":", count, "dry months")


Nairobi : 4 dry months
Nakuru : 3 dry months
Kisumu : 0 dry months
Mombasa : 2 dry months
Machakos : 6 dry months


In [10]:

# Add each month to the following month
two_month_totals = rainfall[:, :-1] + rainfall[:, 1:]

# Find the position of the highest two-month rainfall
planting_indices = np.argmax(two_month_totals, axis=1)

# Get the planting window names
planting_windows = [
    f"{months[index]} - {months[index + 1]}"
    for index in planting_indices
]

# Display the results
for county, window in zip(counties, planting_windows):
    print(county, ":", window)


Nairobi : April - May
Nakuru : April - May
Kisumu : April - May
Mombasa : April - May
Machakos : November - December


In [11]:

# Create the CSV file
with open("rainfall_summary.csv", "w", newline="") as file:

    writer = csv.writer(file)

    # Write the column headings
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    # Write the results for each county
    for county, total, wettest, dry, window in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow([county, total, wettest, dry, window])

print("rainfall_summary.csv created successfully!")


rainfall_summary.csv created successfully!


In [12]:

with open("rainfall_summary.csv", "r") as file:
    print(file.read())


county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April - May
Nakuru,916,April,3,April - May
Kisumu,1371,April,0,April - May
Mombasa,1075,May,2,April - May
Machakos,811,November,6,November - December




## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should consider April to May as their recommended planting window because these two consecutive months have the highest combined rainfall in each county. Farmers in Machakos should consider November to December because this period has the highest combined rainfall for that county. Machakos requires additional attention because it recorded six dry months, the highest among the five counties. Kisumu recorded no dry months and had the highest annual rainfall total of 1,371 mm. Farmers should still monitor local weather forecasts before planting because historical monthly averages do not guarantee future rainfall.



## Data Quality Notes

The original Machakos rainfall data contained two invalid readings: "N/A" in April and an empty value in September. Both values were replaced with 68 mm, calculated from the average of the ten valid Machakos readings and rounded to the nearest whole number. This imputation may affect Machakos rainfall totals and the recommended planting window. The analysis uses the supplied monthly readings and does not account for differences between individual weather stations or future weather conditions.
